# Anatomy of Persuasion · M0 at one hour

**Prediction time:** one hour after a challenger opens a branch. **Outcome:** its first verified OP-awarded delta in the next seven days, among branches without a delta by hour one.

This experiment uses the published training partition for feature extraction, model selection, and evaluation. All time-dependent counts, depth, eligibility rules, and outcomes are rebuilt at one hour with the [shared branch extractor](../../nested-replies-analysis/op_engagement_study/build_branch_dataset.py). The [24-hour experiment](m0_baseline.ipynb) uses a different risk set and base rate. Compare each result to its own prevalence.

M0a uses non-OP comment count, opening delay, and opening length. M0b adds distinct non-OP authors and maximum reply depth. Both use regularized logistic regression.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

from IPython.display import display
import joblib
import numpy as np
import pandas as pd


REPO = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "team-setup").is_dir() and (path / "pyproject.toml").is_file()
)
DATA = REPO / "project/data/processed/m0-baseline/1h"
BUILDER = (
    REPO / "project/notebooks/nested-replies-analysis/op_engagement_study/build_branch_dataset.py"
)
if not all((DATA / "train" / name).is_file() for name in ("branches.parquet", "audit.json")):
    subprocess.run([
        sys.executable, str(BUILDER), "--split", "train", "--model-features", "--observation-hours", "1",
    ], check=True)
audit = json.loads((DATA / "train/audit.json").read_text())
assert (audit["split"], audit["observation_hours"], audit["followup_days_after_cutoff"]) == ("train", 1, 7)
rows = pd.read_parquet(DATA / "train/branches.parquet")
train = rows.loc[rows["eligible"]].sort_values("branch_id").reset_index(drop=True)
TARGET = "delta_after_1h"
train[TARGET] = train[TARGET].astype(int)

# Check whether both OP engagement groups have enough recorded outcomes for a later M1 comparison.
engagement_support = train.groupby("op_engaged_1h").agg(
    branches=("branch_id", "size"), discussions=("post_id", "nunique"),
    later_deltas=(TARGET, "sum"),
).rename(index={False: "No early OP reply", True: "Early OP reply"})
engagement_support["later_delta_rate_pct"] = 100 * engagement_support["later_deltas"] / engagement_support["branches"]
display(engagement_support.round(3))

,branches,discussions,later_deltas,later_delta_rate_pct
op_engaged_1h,,,,
No early OP reply,169506,17598,2358,1.391
Early OP reply,41686,12776,1421,3.409


## Grouped validation inside training

Three outer folds provide out-of-fold predictions, so each branch is scored by a model fitted on other discussions. For each outer fold, three inner folds choose the regularization parameter using average precision. Preprocessing learns median imputation and scaling inside each fit. The two models use the same folds and class frequencies.

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from threadpoolctl import threadpool_limits


SEED = 42
FEATURES = {
    "M0a": ["non_op_comments_1h", "opening_delay_minutes", "opening_reply_length"],
    "M0b": ["non_op_comments_1h", "opening_delay_minutes", "opening_reply_length",
            "unique_non_op_authors_1h", "max_depth_1h"],
}
CANDIDATE_C = [0.01, 0.1, 1, 10]


def make_search(seed):
    """Use the same preprocessing and regularization grid as the 24-hour baselines."""
    pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p)),
        ("scale", StandardScaler()),
        ("logistic", LogisticRegression(solver="lbfgs", max_iter=1000)),
    ])
    return GridSearchCV(
        pipeline, {"logistic__C": CANDIDATE_C}, scoring="average_precision",
        cv=StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=seed),
        refit=True, error_score="raise",
    )


predictions = train[["post_id", "branch_id", TARGET]].copy()
predictions["fold"] = -1
for name in FEATURES:
    predictions[name] = np.nan
outer = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
fold_metrics = []
with threadpool_limits(limits=1):
    for fold, (fit_index, eval_index) in enumerate(
        outer.split(train, train[TARGET], groups=train["post_id"]), start=1
    ):
        fitting, validation = train.iloc[fit_index], train.iloc[eval_index]
        assert set(fitting["post_id"]).isdisjoint(validation["post_id"])
        predictions.loc[eval_index, "fold"] = fold
        for name, columns in FEATURES.items():
            search = make_search(SEED + fold)
            search.fit(fitting[columns], fitting[TARGET], groups=fitting["post_id"])
            probability = search.predict_proba(validation[columns])[:, 1]
            predictions.loc[eval_index, name] = probability
            fold_metrics.append({
                "fold": fold, "model": name, "C": search.best_params_["logistic__C"],
                "branches": len(validation), "later deltas": validation[TARGET].sum(),
                "prevalence_pct": 100 * validation[TARGET].mean(),
                "average_precision_pct": 100 * average_precision_score(validation[TARGET], probability),
            })
assert predictions["fold"].gt(0).all() and predictions[list(FEATURES)].notna().all().all()
assert predictions.groupby("post_id")["fold"].nunique().eq(1).all()
display(pd.DataFrame(fold_metrics).set_index(["fold", "model"]).round(4))

C  branches  later deltas  prevalence_pct  average_precision_pct
fold model                                                                   
1    M0a    10     70397          1259          1.7884                 4.2220
     M0b     1     70397          1259          1.7884                 4.3755
2    M0a    10     70399          1261          1.7912                 4.4810
     M0b    10     70399          1261          1.7912                 4.5615
3    M0a    10     70396          1259          1.7885                 4.6510
     M0b    10     70396          1259          1.7885                 4.7229

In [3]:
# Compare pooled out-of-fold predictions to the prevalence of the one-hour risk set.
y = predictions[TARGET].to_numpy()
prevalence = y.mean()
summary_rows = []
for name in FEATURES:
    probability = predictions[name].to_numpy()
    ranked = np.argsort(-probability, kind="stable")[:100]
    ap = average_precision_score(y, probability)
    summary_rows.append({
        "model": name, "average_precision_pct": 100 * ap, "AP / prevalence": ap / prevalence,
        "ROC-AUC": roc_auc_score(y, probability), "Brier score": brier_score_loss(y, probability),
        "deltas in top 100": int(y[ranked].sum()),
    })
summary = pd.DataFrame(summary_rows).set_index("model")
print(f"{len(y):,} eligible training branches · {int(y.sum()):,} later deltas · {prevalence:.3%} prevalence")
display(summary.round(5))
predictions.to_parquet(DATA / "training_oof_predictions.parquet", index=False)

211,192 eligible training branches · 3,779 later deltas · 1.789% prevalence


,average_precision_pct,AP / prevalence,ROC-AUC,Brier score,deltas in top 100
model,,,,,
M0a,4.40788,2.46337,0.72358,0.01735,6
M0b,4.52055,2.52634,0.72818,0.01734,6


## Refit on the full training split

After the out-of-fold evaluation, tune and refit the two pipelines on all eligible training branches. Save the fitted pipelines and their feature lists for a later M1 comparison. The validation metrics above come from the out-of-fold predictions.

In [4]:
final_settings = {}
with threadpool_limits(limits=1):
    for name, columns in FEATURES.items():
        search = make_search(SEED)
        search.fit(train[columns], train[TARGET], groups=train["post_id"])
        path = DATA / f"{name}.joblib"
        joblib.dump(search.best_estimator_, path)
        final_settings[name] = {
            "features": columns, "C": search.best_params_["logistic__C"], "artifact": path.name,
        }

(DATA / "training_model_settings.json").write_text(
    json.dumps({
        "split": "train", "observation_hours": 1, "followup_days": 7,
        "training_branches": len(train), "training_later_deltas": int(train[TARGET].sum()),
        "models": final_settings,
    }, indent=2) + "\n"
)
display(pd.DataFrame(final_settings).T)

,features,C,artifact
M0a,"[non_op_comments_1h, opening_delay_minutes, opening_reply_length]",10,M0a.joblib
M0b,"[non_op_comments_1h, opening_delay_minutes, opening_reply_length, unique_non_op_authors_1h, max_depth_1h]",10,M0b.joblib
